![alt text](./Cerny_logo_1.jpg)

# Analysis of Cerny ventilation recordings

#### Processing blood gases

This notebook imports and processes blood gas data and exports it into a pickle archive.

The data processed and analysed in this Notebook were collected by the **Neonatal Emergency and Transport Service of the Peter Cerny Foundation**, Budapest, Hungary

**Author: Dr Gusztav Belteki**


- Recordings with >=5 minutes of ventilator data: **1966 cases**
- Clinical data abd ventilation data (>=5 minutes) are available: **1880 cases**.
- Blood gases is also available in **1771 cases**

### 1. Import the required libraries and set options

In [5]:
import IPython
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib
import matplotlib.pyplot as plt

import os
import sys
import re
import pickle

from scipy import stats
from pandas import Series, DataFrame
from datetime import datetime, timedelta

%matplotlib inline
matplotlib.style.use('classic')
matplotlib.rcParams['figure.facecolor'] = 'w'

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('mode.chained_assignment', None) 

# This is to turn off a warning message which is given when read_Excel() imports '.xlsx' files
import warnings
warnings.simplefilter("ignore")

In [6]:
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("matplotlib version: {}".format(matplotlib.__version__))
print("NumPy version: {}".format(np.__version__))
print("SciPy version: {}".format(sp.__version__))
print("IPython version: {}".format(IPython.__version__))

Python version: 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:38:53) [Clang 18.1.8 ]
pandas version: 2.3.2
matplotlib version: 3.10.6
NumPy version: 1.26.4
SciPy version: 1.16.2
IPython version: 9.5.0


### 2. List and set the working directory and the directory to write out data

In [8]:
# Name of the external hard drive
DRIVE = 'GB_1'

# Directory on external drive to read the clinical from
DIR_READ = os.path.join(os.sep, 'Volumes', DRIVE, 'Ventilator_data', 'Fabian_new', 'fabian_patient_data_all_new')

# Path to project folder containing ventilation research results
PATH = os.path.join(os.sep, 'Users', 'guszti', 'Library', 'Mobile Documents', 'com~apple~CloudDocs', 
                            'Documents', 'Research', 'Ventilation')

# Folder to export the result of analysis
DIR_WRITE = os.path.join(PATH, 'ventilation_fabian_new', 'Analyses')
os.makedirs(DIR_WRITE, exist_ok = True)

# Folder on a USB stick to export data to and to import processed data exported by other Notebooks
DATA_DUMP = os.path.join(os.sep, '/Volumes', DRIVE, 'data_dump', 'fabian_new',)
os.makedirs(DATA_DUMP, exist_ok = True)

In [9]:
DIR_READ, DIR_WRITE, DATA_DUMP

('/Volumes/GB_1/Ventilator_data/Fabian_new/fabian_patient_data_all_new',
 '/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian_new/Analyses',
 '/Volumes/GB_1/data_dump/fabian_new')

### 3. Import clinical DataFrame from pickle archive

In [11]:
with open(os.path.join(DATA_DUMP, 'clin_df_new.pickle'), 'rb') as handle:
    clin_df = pickle.load(handle)

In [12]:
cases = sorted(clin_df.index)

In [13]:
len(cases)

1880

### 4. Import all clinical data containing blood gases

In [15]:
# import text files in a dictionary
clin_dict = {}
for fname in os.listdir(DIR_READ):
    if not fname.startswith('.'): # disregard hidden files
        #print(fname)
        fhandle = open(os.path.join(DIR_READ, fname), 'r', encoding = 'cp1252', errors='ignore')
        clin_dict[fname[:-4]] = fhandle.read() # use the filenames without the .txt extension as keys
        fhandle.close()

In [16]:
len(clin_dict)

1937

In [17]:
# Limit clinical data to recordings from which ventilator data are available
clin_dict = {key: value for key, value in clin_dict.items() if key in cases }

In [18]:
len(clin_dict)

1880

In [19]:
gas_dict = {}
# Remove clinical details preceding the blood gases
for key, value in clin_dict.items():
    # For recordings starting with AT001263, 'Astrup' in the text file was changed to 'Labor'
    if 'Astrup' in value:
        gas_dict[key] = value[value.index('Astrup'):]
    elif 'Labor' in value:
        gas_dict[key] = value[value.index('Labor'):]
    else:
        print(key, 'has no blood gas')

AT002070 has no blood gas
AT002084 has no blood gas
AT002086 has no blood gas
AT002091 has no blood gas
AT002092 has no blood gas
AT001299 has no blood gas
AT001281 has no blood gas
AT001372 has no blood gas
AT001403 has no blood gas
AT001404 has no blood gas
AT001415 has no blood gas
AT001494 has no blood gas
AT001534 has no blood gas
AT001605 has no blood gas
AT001609 has no blood gas
AT001610 has no blood gas
AT001615 has no blood gas
AT001626 has no blood gas
AT001628 has no blood gas
AT001635 has no blood gas
AT002103 has no blood gas
AT001672 has no blood gas
AT001690 has no blood gas
AT001700 has no blood gas
AT001789 has no blood gas
AT001814 has no blood gas
AT001847 has no blood gas
AT001898 has no blood gas
AT001885 has no blood gas
AT001924 has no blood gas
AT001974 has no blood gas
AT001464 has no blood gas
AT001529 has no blood gas
AT001586 has no blood gas
AT001748 has no blood gas
AT001842 has no blood gas
AT001881 has no blood gas
AT002109 has no blood gas
AT002159 has

In [20]:
len(gas_dict)

1837

In [21]:
gas_dict_2 = {}

for key, value in gas_dict.items():
    #if key in ['AT001273', 'AT001280', 'AT001283', 'AT001287', 'AT001297', 'AT001688', 'AT001717']:
    #    continue
    gas_dict_2[key] = {}
    # Recordings before and after AT001263 have different formats and they need to be processed differently 
    (wrd := 'Astrup') if int(key[2:].lstrip('0')) < 1263 else (wrd := 'Labor')
    for i, gas in enumerate(value.split(wrd)[1:]):
        gas_dict_2[key][i] = {}
        items = gas.split('\n')[1:-1]
        for item in items:
            try:
                # keys and values are separated by ':', some values contain ':' (timestamps), ignore those
                name, value = item.split(':', maxsplit=1) 
                if value.strip() == '':
                    continue
                else:
                    # The split is needed to remove units from the values
                    if name.strip() != 'Time':
                        value = value.split(' ')[1]
                    gas_dict_2[key][i][name.strip()] = value.lstrip()

            except:
                pass        

In [22]:
len(gas_dict_2)

1837

In [23]:
for case in gas_dict_2:
    for gas in sorted(gas_dict_2[case].keys()):
        if gas_dict_2[case][gas] == {}:
            del gas_dict_2[case][gas]

In [24]:
gas_frames = {}
for case in gas_dict_2.keys():
    gas_frames[case] = DataFrame(gas_dict_2[case])

In [25]:
for rec in gas_frames:
    if int(rec[2:].lstrip('0')) < 1263:
        dte = clin_df.loc[rec]['Recording start'].date()
        lasttme = ''
        for column in gas_frames[rec]:
            tme = datetime.strptime(str(gas_frames[rec][column]['Time']), '%H%M').time()
            # tme is after midnight, lasttme is before midnight
            if lasttme and lasttme > tme:
                dte = dte + timedelta(1) # increase date by 1 day
            gas_frames[rec][column]['Time'] = datetime.combine(dte, tme)
            lasttme = tme
    else:
        for column in gas_frames[rec]:
            gas_frames[rec][column]['Time'] = pd.to_datetime(gas_frames[rec][column]['Time'])

In [26]:
for rec in gas_frames:
    if int(rec[2:].lstrip('0')) >= 1263:
        gas_frames[rec].rename({'Sample site':'Type', 'cSO2':'Saturatio', 'BE(ecf)':'ABE'}, axis=1, inplace=True)
        gas_frames[rec].replace({'KapillÃ¡ris':'capillaris', 'VÃ©nÃ¡s':'Venas'}, inplace=True)

In [27]:
gas_frames['AT000935']

,0,1,2
Time,2023-01-09 02:35:00,2023-01-09 03:14:00,2023-01-09 05:44:00
pH,6.867,7.023,7.287
pCO2,98.2,82.8,60.8
pO2,37.6,45.1,36.3
HCO3,17.8,21.5,28.4
ABE,-15.6,-9.4,0.4
Saturatio,35.5,57.5,83.1
FiO2,0.60,0.90,0.38
Type,Capillaris,Capillaris,Capillaris


In [28]:
gas_frames['AT001285']

,0,1
Time,2024-04-25 05:18:00,2024-04-25 06:30:00
Sample site,capillaris,capillaris
Hypothermia,nem,nem
VÃ©rcukor,4.4,3.7
pH,7.18,7.37
pCO2,44,36
pO2,48,59
HCO3,16,21
BE(ecf),-11.5,-3.9
cSO2,74,96


In [29]:
for case in sorted(gas_frames.keys()):
    try:
        gas_frames[case] =  gas_frames[case].T.set_index('Time')
    except:
        print('No blood gas for %s' % case)
        del gas_frames[case]

No blood gas for AT000007
No blood gas for AT000020
No blood gas for AT000022
No blood gas for AT000030
No blood gas for AT000038
No blood gas for AT000039
No blood gas for AT000040
No blood gas for AT000052
No blood gas for AT000057
No blood gas for AT000070
No blood gas for AT000078
No blood gas for AT000103
No blood gas for AT000133
No blood gas for AT000134
No blood gas for AT000136
No blood gas for AT000140
No blood gas for AT000146
No blood gas for AT000151
No blood gas for AT000155
No blood gas for AT000179
No blood gas for AT000180
No blood gas for AT000200
No blood gas for AT000213
No blood gas for AT000250
No blood gas for AT000251
No blood gas for AT000291
No blood gas for AT000301
No blood gas for AT000315
No blood gas for AT000351
No blood gas for AT000359
No blood gas for AT000397
No blood gas for AT000408
No blood gas for AT000416
No blood gas for AT000451
No blood gas for AT000495
No blood gas for AT000517
No blood gas for AT000528
No blood gas for AT000534
No blood gas

In [30]:
len(gas_frames)

1771

### 5. Quality control of blood gases

#### Combine all gases to a single DataFrame

In [33]:
gas_frames_all = pd.concat(gas_frames)

In [34]:
gas_frames_all = gas_frames_all[['pH', 'pCO2', 'pO2', 'HCO3', 'ABE', 'FiO2', 'Type']]
for column in ['pH', 'pCO2', 'pO2', 'HCO3', 'ABE', 'FiO2']:
    gas_frames_all[column] = gas_frames_all[column].dropna().astype('float', errors='ignore')
for column in ['Type']:
    gas_frames_all[column] = gas_frames_all[column].astype('category')

In [35]:
gas_frames_all[gas_frames_all['pO2'] == '']

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,


In [36]:
gas_frames_all.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 3353 entries, ('AT001688', Timestamp('2025-03-15 15:36:00')) to ('AT002334', Timestamp('2026-07-03 12:01:00'))
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   pH      3288 non-null   float64 
 1   pCO2    3299 non-null   float64 
 2   pO2     3244 non-null   float64 
 3   HCO3    3240 non-null   float64 
 4   ABE     1732 non-null   float64 
 5   FiO2    1708 non-null   float64 
 6   Type    1700 non-null   category
dtypes: category(1), float64(6)
memory usage: 407.0+ KB


#### Manually review outlier with outlier values in gases and correct as appropriate

- pH <6.6 or pH >7.5
- pCO2 <20 mmHg or >130 mmHg
- ABE <-30 or >20 

Only correct trivial ones. 

- For example, "7" is sometimes mis-recognised as "2" by OCR. 
- Other times, the decimal point is clearly in the wrong place.

Only correct those ones where the other values in the blood gas are consistent with the change you are making. Otherwise, remove the clearly impossible values.

In [38]:
# These are all correct.
gas_frames_all[gas_frames_all['pH'] < 6.6].sort_values('pH', ascending=True)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,
AT000820,2022-08-12 17:22:00,6.500,160.0,20.0,NaN,NaN,1.00,Capillaris
AT002197,2026-03-22 19:27:00,6.500,104.0,60.0,NaN,NaN,NaN,NaN
AT000189,2021-03-25 17:13:00,6.500,188.7,40.5,NaN,NaN,0.21,Capillaris
AT000639,2022-03-02 15:08:00,6.500,110.6,137.3,NaN,NaN,1.00,Capillaris
AT001618,2024-12-29 04:12:00,6.500,101.8,40.1,8.4,NaN,NaN,NaN
AT000820,2022-08-12 16:40:00,6.500,157.0,31.4,NaN,NaN,1.00,Capillaris
AT002222,2026-04-18 19:50:00,6.500,153.1,185.0,NaN,NaN,NaN,NaN
AT001801,2025-05-28 19:27:00,6.516,141.7,58.7,11.5,NaN,NaN,NaN
AT002197,2026-03-22 18:30:00,6.520,144.0,12.0,12.0,NaN,NaN,NaN


In [39]:
# These are all correct
gas_frames_all[gas_frames_all['pH'] > 7.5].sort_values('pH', ascending=False)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,
AT001714,2025-03-31 18:54:00,7.700,34.8,46.6,43.5,NaN,NaN,NaN
AT000598,2022-01-19 10:55:00,7.634,24.8,42.3,26.4,5.4,0.21,Capillaris
AT000913,2022-12-25 12:05:00,7.623,19.5,14.0,20.4,-10.0,0.21,Capillaris
AT001808,2025-06-04 17:18:00,7.598,23.4,42.0,22.8,NaN,NaN,NaN
AT001381,2024-06-25 09:51:00,7.556,39.8,90.3,35.3,NaN,NaN,NaN
AT001039,2023-03-25 14:05:00,7.556,31.2,49.9,27.7,5.4,0.25,Capillaris
AT002325,2026-06-29 16:18:00,7.540,47.4,47.2,40.2,NaN,NaN,NaN
AT002171,2026-02-18 18:53:00,7.537,37.7,34.0,32.1,NaN,NaN,NaN
AT002040,2025-11-27 15:33:00,7.533,37.3,40.9,31.4,NaN,NaN,NaN


In [40]:
# These are all correct
gas_frames_all.dropna(how='any')[gas_frames_all.dropna(how='any')['pCO2'] > 130].sort_values('pCO2', ascending=True)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,
AT000693,2022-04-21 17:07:00,6.866,133.4,31.7,24.2,-11.5,0.3,Capillaris
AT000767,2022-07-03 16:41:00,6.525,134.0,13.7,10.4,-23.5,1.0,Venas
AT000591,2022-01-14 08:37:00,6.825,143.6,46.1,23.7,-10.4,1.0,Capillaris


In [41]:
# These are all correct
gas_frames_all[gas_frames_all['pCO2'] < 20].sort_values('pCO2', ascending=True)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,
AT001418,2024-07-15 15:32:00,7.338,18.1,106.0,9.4,NaN,NaN,NaN
AT001657,2025-01-30 14:11:00,7.000,18.2,57.7,6.4,NaN,NaN,NaN
AT000450,2021-10-08 20:30:00,7.380,19.5,73.3,11.5,-13.6,0.21,Capillaris
AT000913,2022-12-25 12:05:00,7.623,19.5,14.0,20.4,-10.0,0.21,Capillaris


In [42]:
# These are all correct
gas_frames_all[gas_frames_all['ABE'] < - 30].sort_values('ABE', ascending=True)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,
AT000099,2021-01-13 20:00:00,6.548,52.4,49.9,4.5,-34.1,1.00,Venas
AT001181,2023-07-17 06:00:00,6.520,105.0,61.0,8.4,-34.0,0.40,Venas
AT000912,2022-12-23 18:55:00,6.650,86.7,90.3,9.4,-31.0,0.21,Capillaris


In [43]:
# These are all correct
gas_frames_all[gas_frames_all['ABE'] > 20].sort_values('ABE', ascending=True)

,,pH,pCO2,pO2,HCO3,ABE,FiO2,Type
,Time,,,,,,,
AT000815,2022-08-05 21:40:00,7.410,28.0,43.0,-6.2,20.8,1.00,Venas
AT000637,2022-03-01 06:10:00,7.360,82.0,20.0,46.3,20.9,1.00,Venas
AT000230,2021-05-02 19:48:00,7.360,42.8,38.6,-1.3,22.0,NaN,Capillaris
AT000876,2022-11-03 16:53:00,7.360,93.0,21.0,41.6,23.0,0.60,Capillaris
AT000862,2022-10-27 12:30:00,7.510,59.0,40.0,47.0,24.1,NaN,NaN
AT000952,2023-01-17 22:50:00,7.370,96.0,62.0,55.0,25.0,1.00,Capillaris
AT000381,2021-08-27 14:47:00,7.269,64.4,45.9,29.5,26.0,0.30,Capillaris
AT000952,2023-01-17 18:38:00,7.480,66.0,49.0,50.0,27.0,1.00,Capillaris
AT000510,2021-11-21 13:06:00,7.400,47.0,52.0,29.0,36.0,0.30,Capillaris


### 6. Export bood gases as Excel files

### 7. Export processed data as pickle files

In [46]:
with open(os.path.join(DATA_DUMP, 'blood_gases_new.pickle'), 'wb') as handle:
    pickle.dump(gas_frames, handle, protocol=pickle.HIGHEST_PROTOCOL)